In [0]:
USE CATALOG nasa_analytics;

In [0]:
CREATE OR REPLACE TABLE gold.donki_event_counts AS
SELECT
    date_trunc('month', issue_time) AS event_month,
    message_type,
    COUNT(*) AS event_count
FROM silver.donki_notifications
GROUP BY date_trunc('month', issue_time), message_type
ORDER BY event_month, message_type;

In [0]:
CREATE OR REPLACE TABLE gold.donki_latest_alerts AS
SELECT
    message_type,
    MAX(issue_time) AS latest_issue_time
FROM silver.donki_notifications
GROUP BY message_type;

In [0]:
from pyspark.sql.functions import when, lower

df = spark.table("nasa_analytics.silver.donki_notifications")

df_severity = df.withColumn(
    "severity",
    when(lower(col("message_body")).like("%strong%"), "High")
    .when(lower(col("message_body")).like("%moderate%"), "Medium")
    .when(lower(col("message_body")).like("%minor%"), "Low")
    .otherwise("Unknown")
)

df_severity.write.mode("overwrite").saveAsTable("nasa_analytics.gold.donki_event_severity")

In [0]:
SELECT * FROM gold.donki_event_counts LIMIT 20;

SELECT * FROM gold.donki_latest_alerts;

SELECT message_type, severity, COUNT(*) 
FROM gold.donki_event_severity
GROUP BY message_type, severity;